In [1]:
import os
import re
import time
import requests
import numpy as np
import pandas as pd

from dotenv import load_dotenv

In [2]:
# 데이터 로드
RAW_PATH = "../data/roomhubs_seoul_raw_20260128_144910.csv"
ST_PATH  = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"
CODE_PATH = "../data/sigungu_code_info.csv"

df_raw = pd.read_csv(RAW_PATH)

# 서교공 파일은 EUC-KR 인코딩
st_raw = pd.read_csv(ST_PATH, encoding="euc-kr")

print(df_raw.shape, df_raw.columns.tolist())
print(st_raw.shape, st_raw.columns.tolist())

(23, 10) ['external_id', 'source_site', 'crawled_at', 'listing_url', 'detail_url', 'name_raw', 'category_raw', 'address_raw', 'map_url_raw', 'status_raw']
(289, 7) ['연번', '역번호', '호선', '역명', '역전화번호', '도로명주소', '지번주소']


In [3]:
#주소 표준화
def normalize_spaces(x: str) -> str:
    x = str(x).strip()
    x = re.sub(r"[《》]", " ", x)          # 특수 괄호류 제거
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_seoul_prefix(addr: str) -> str:
    """
    '서울', '서울시' -> '서울특별시'로 통일
    """
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return addr
    a = normalize_spaces(addr)
    a = re.sub(r"^서울시\s+", "서울특별시 ", a)
    a = re.sub(r"^서울\s+", "서울특별시 ", a)
    return a

In [4]:
# 역 매핑 테이블 생성
def clean_station_name(name: str) -> str:
    """
    서교공 역명: '종로3가' -> '종로3가역'
    """
    n = normalize_spaces(name)
    n = re.sub(r"\(.*?\)", "", n).strip()  # 혹시 괄호 부가정보가 있으면 제거
    if not n.endswith("역"):
        n += "역"
    return n

st = st_raw.copy()

st["station_std"] = st["역명"].apply(clean_station_name)

# 서교공 도로명주소는 대개 '서울특별시 ...' 형태지만, 안전하게 prefix normalize
st["station_road_addr"] = st["도로명주소"].astype(str).apply(normalize_seoul_prefix)

# 대표 주소 선택(환승역 중복 제거)
st_master = (
    st.groupby("station_std", as_index=False)
      .agg(station_road_addr=("station_road_addr", "first"))
)

station_road_dict = dict(zip(st_master["station_std"], st_master["station_road_addr"]))
station_master_list = st_master["station_std"].tolist()

print("서교공 역 개수(중복 제거):", len(st_master))
display(st_master.sample(10, random_state=1))

서교공 역 개수(중복 제거): 251


,station_std,station_road_addr
67,디지털미디어시티역,서울특별시 은평구 수색로 지하175(증산동)
250,효창공원앞역,서울특별시 용산구 백범로 지하287(효창동)
230,충정로역,서울특별시 서대문구 서소문로 지하17(충정로3가)
161,안암역,서울특별시 성북구 고려대로 지하89(안암동5가)
91,발산역,서울특별시 강서구 공항대로 지하267(마곡동)
224,철산역,경기도 광명시 철산로 지하13(철산동)
58,독바위역,서울특별시 은평구 불광로 지하129-1(불광동)
234,하남시청역,경기도 하남시 하남대로 지하820(덕풍동)
180,오목교역,서울특별시 양천구 오목로 지하342(목동)
4,강동구청역,서울특별시 강동구 올림픽로 지하550(성내동)


In [5]:
df = df_raw.copy()

# name/category는 raw 그대로
df["name"] = df["name_raw"]
df["category"] = df["category_raw"]

def looks_like_address(x: str) -> bool:
    # 이 파일은 대체로 '서울 강남구 ...' 또는 '서울특별시 ...' 형태
    return bool(re.match(r"^(서울|서울시|서울특별시)\s+\S+구\s+", x))

def split_address_station(raw: str):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return (np.nan, np.nan)

    raw = normalize_spaces(raw)

    # 주소가 아닌 케이스: 역 정보만
    if not looks_like_address(raw):
        return (np.nan, raw)

    # '서울 ... (삼성동) 삼성중앙역 ...' 패턴
    if "(" in raw and ")" in raw and raw.index("(") < raw.index(")"):
        before = raw[:raw.index("(")].strip()
        after  = raw[raw.index(")")+1:].strip()
    else:
        before = raw.strip()
        after = ""

    # station_detail 정리: '서울,경기,인천' 같은 서비스 범위 문자열 제거
    if after:
        # 역/출구/도보 등의 키워드가 없고 지역 나열이면 제거
        if (not re.search(r"(역|출구|도보|인근|거리)", after)) and re.search(r"(서울|경기|인천)", after):
            after = ""

    address_std = normalize_seoul_prefix(before) if before else np.nan
    station_detail = after if after else np.nan

    return (address_std, station_detail)

df[["address_std", "station_detail"]] = df["address_raw"].apply(lambda x: pd.Series(split_address_station(x)))

display(df[["address_raw","address_std","station_detail"]].head(15))


,address_raw,address_std,station_detail
0,서울시 강남구 역삼동 604-11,서울특별시 강남구 역삼동 604-11,NaN
1,서울 관악구 신림동 1640-30,서울특별시 관악구 신림동 1640-30,NaN
2,서울 강남구 봉은사로 150,서울특별시 강남구 봉은사로 150,NaN
3,서울 서초구 서초대로78길 46,서울특별시 서초구 서초대로78길 46,NaN
4,서울 강남구 테헤란로 111 대건빌딩 지하1층,서울특별시 강남구 테헤란로 111 대건빌딩 지하1층,NaN
5,서울 등촌역삼거리,NaN,서울 등촌역삼거리
6,서울시 송파구 가락동 74,서울특별시 송파구 가락동 74,NaN
7,서울시 송파구 오금로11길 29-14,서울특별시 송파구 오금로11길 29-14,NaN
8,서울 강남구 테헤란로 111,서울특별시 강남구 테헤란로 111,NaN
9,서울시 강남구 삼성동 145,서울특별시 강남구 삼성동 145,NaN


In [6]:
# is_valid_location 판단

def is_station_based_raw(raw: str):
    """
    raw 주소가 실제 업소 주소가 아니라
    '신림역 2번출구 도보2분' 같은 역 기반 설명이면 True 반환
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return False

    t = normalize_spaces(raw)

    # 1) 아예 주소 형식이 아닌 경우
    if not looks_like_address(t):
        return True

    # 2) 주소는 있는데 괄호 뒤가 전부 역 설명인 경우
    if re.search(r"(역|출구|도보)", t) and not re.search(r"\d+로|\d+길|\d+-\d+", t):
        # 도로명 숫자 패턴이 없고 역 키워드가 중심이면
        return True

    return False


df["is_station_based_raw"] = df["address_raw"].apply(is_station_based_raw)

print("역 기반 raw 주소 개수:", df["is_station_based_raw"].sum())


역 기반 raw 주소 개수: 4


In [7]:
# station_exit 및 station 파싱
def find_station_by_master(text: str):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return None
    t = normalize_spaces(text)

    # 포함되는 역 후보 찾기
    matches = [s for s in station_master_list if s in t]
    if not matches:
        return None

    # 가장 먼저 등장하는 역을 채택
    matches.sort(key=lambda s: t.index(s))
    return matches[0]

def parse_station_fields_with_master(station_detail):
    if station_detail is None or (isinstance(station_detail, float) and np.isnan(station_detail)):
        return (np.nan, np.nan)

    t = normalize_spaces(station_detail)

    # 도보 n분 제거(출구/역명만 남김)
    t2 = re.sub(r"도보\s*\d+\s*분", "", t).strip()

    station = find_station_by_master(t2)
    if not station:
        return (np.nan, np.nan)

    m_exit = re.search(r"(\d+)\s*번\s*출구", t2)
    if m_exit:
        station_exit = f"{station} {m_exit.group(1)}번출구"
    else:
        station_exit = np.nan

    return (station_exit, station)

df[["station_exit", "station"]] = df["station_detail"].apply(lambda x: pd.Series(parse_station_fields_with_master(x)))

display(df[["station_detail","station_exit","station"]].sample(20, random_state=2))

,station_detail,station_exit,station
20,NaN,NaN,NaN
0,NaN,NaN,NaN
21,NaN,NaN,NaN
6,NaN,NaN,NaN
14,NaN,NaN,NaN
16,NaN,NaN,NaN
3,NaN,NaN,NaN
12,NaN,NaN,NaN
9,NaN,NaN,NaN
4,NaN,NaN,NaN


In [8]:
# 자치구 파싱
def parse_gu_from_addr(addr: str):
    if addr is None or (isinstance(addr, float) and np.isnan(addr)):
        return np.nan
    a = normalize_seoul_prefix(addr)
    m = re.search(r"서울특별시\s+(\S+구)\b", a)
    if m:
        return m.group(1)
    # 혹시 '서울 강남구' 형태가 남아있다면
    m2 = re.search(r"서울\s+(\S+구)\b", a)
    if m2:
        return m2.group(1)
    return np.nan

# 1차 gu 채우기(상세주소가 있으면 주소에서)
df["gu"] = df["address_std"].apply(parse_gu_from_addr)

In [9]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY[:6] + "..." if GOOGLE_API_KEY else "❌ 키 로드 실패")

AIzaSy...


In [10]:
# geocoding fuction
def geocode_google(query: str, api_key: str):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": query,
        "key": api_key,
        "language": "ko",
        "region": "kr",
    }
    r = requests.get(url, params=params, timeout=20)
    data = r.json()

    if data.get("status") != "OK" or not data.get("results"):
        return (None, None, None, None)

    top = data["results"][0]
    loc = top["geometry"]["location"]
    lat, lng = loc["lat"], loc["lng"]

    # gu 추출(가능한 범위 내)
    gu = None
    for comp in top.get("address_components", []):
        if "sublocality_level_1" in comp.get("types", []):
            gu = comp.get("long_name")
            break
        if "administrative_area_level_2" in comp.get("types", []):
            gu = comp.get("long_name")

    return (lat, lng, gu, top.get("formatted_address"))

cache = {}

def geocode_cached(query: str, sleep_sec=0.1):
    if query is None or (isinstance(query, float) and np.isnan(query)):
        return (None, None, None, None)
    q = normalize_spaces(query)
    if not q:
        return (None, None, None, None)
    if q in cache:
        return cache[q]

    out = geocode_google(q, GOOGLE_API_KEY)
    cache[q] = out
    time.sleep(sleep_sec)
    return out

def build_station_exit_query(station_exit: str):
    # 서울로 유도
    return f"서울특별시 {station_exit}" if station_exit else None

def build_station_query(station: str):
    # 1) 서교공 도로명주소가 있으면 그걸 최우선 사용
    if station in station_road_dict:
        return station_road_dict[station]
    # 2) 없으면 '서울특별시 + 역명'으로 보완
    return f"서울특별시 {station}" if station else None

In [11]:
# run geocoding

df["lat"] = np.nan
df["lng"] = np.nan
df["COL_ADM_SE"] = np.nan

# is_valid_location은 지오코딩 성공 여부가 아니라
# "실제 업소 주소인지 여부"로 먼저 결정
df["is_valid_location"] = ~df["is_station_based_raw"]

# 1) 실제 업소 주소인 경우만 address_std로 1차 지오코딩
for i, row in df.iterrows():
    if row["is_valid_location"] and pd.notna(row["address_std"]):
        lat, lng, gu_geo, _ = geocode_cached(row["address_std"])
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 2) 역 기반인 경우는 항상 is_valid_location=False 유지
#    station_exit → station fallback 수행

mask_station_based = df["is_valid_location"] == False

# station_exit 우선
for i, row in df[mask_station_based].iterrows():
    if pd.notna(row["station_exit"]):
        q = build_station_exit_query(row["station_exit"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo

# 여전히 실패 → station (서교공 도로명주소)
mask_need2 = (df["is_valid_location"] == False) & (df["lat"].isna())

for i, row in df[mask_need2].iterrows():
    if pd.notna(row["station"]):
        q = build_station_query(row["station"])
        lat, lng, gu_geo, _ = geocode_cached(q)
        if lat is not None:
            df.at[i, "lat"] = lat
            df.at[i, "lng"] = lng

            if pd.isna(df.at[i, "gu"]):
                df.at[i, "gu"] = parse_gu_from_addr(q)
            if pd.isna(df.at[i, "gu"]) and gu_geo:
                df.at[i, "gu"] = gu_geo


In [12]:
# gu 최종 보완
# gu가 남아있으면 raw에서라도 파싱 시도
mask_gu_missing = df["gu"].isna()
df.loc[mask_gu_missing, "gu"] = df.loc[mask_gu_missing, "address_raw"].apply(parse_gu_from_addr)

print("gu 결측 개수:", df["gu"].isna().sum())
display(df[df["gu"].isna()][["name","address_raw","address_std","station_detail","station_exit","station"]].head(30))

gu 결측 개수: 1


,name,address_raw,address_std,station_detail,station_exit,station
5,다국적 노래클럽,서울 등촌역삼거리,NaN,서울 등촌역삼거리,NaN,NaN


In [13]:
# COL_ADM_SE 매핑
code_df = pd.read_csv(CODE_PATH, encoding="utf-8", dtype=str)

# 서울(시도코드 11)만 필터
seoul_gu = code_df[code_df["SIGUNGU_CD"].str.startswith("11")].copy()

# 매핑 딕셔너리 생성
gu_code_dict = dict(zip(seoul_gu["SIGUNGU_NM"], seoul_gu["SIGUNGU_CD"]))

# gu 정규화
def normalize_gu(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    # 혹시 "서울특별시 강남구" 형태라면 마지막 토큰만 사용
    if " " in x:
        x = x.split()[-1]
    return x

df["gu"] = df["gu"].apply(normalize_gu)

# 코드 매핑
df["COL_ADM_SE"] = df["gu"].map(gu_code_dict)

# 점검 출력
print("COL_ADM_SE 결측:", df["COL_ADM_SE"].isna().sum())

display(
    df[df["COL_ADM_SE"].isna()][
        ["name","address_std","gu"]
    ].head(20)
)


COL_ADM_SE 결측: 1


,name,address_std,gu
5,다국적 노래클럽,NaN,NaN


In [14]:
df_false = df[df["is_valid_location"] == False].copy()

print("is_valid_location=False 개수:", len(df_false))
print("그 중 address_std 존재:", df_false["address_std"].notna().sum())
print("그 중 station_exit 존재:", df_false["station_exit"].notna().sum())
print("그 중 station 존재:", df_false["station"].notna().sum())
print("그 중 lat 결측:", df_false["lat"].isna().sum())

display(df_false[["name","address_raw","address_std","station_detail","station_exit","station","lat","lng","gu"]].head(30))


is_valid_location=False 개수: 4
그 중 address_std 존재: 2
그 중 station_exit 존재: 0
그 중 station 존재: 0
그 중 lat 결측: 4


,name,address_raw,address_std,station_detail,station_exit,station,lat,lng,gu
5,다국적 노래클럽,서울 등촌역삼거리,NaN,서울 등촌역삼거리,NaN,NaN,NaN,NaN,NaN
11,신림다국적,서울 관악구 신림역사거리,서울특별시 관악구 신림역사거리,NaN,NaN,NaN,NaN,NaN,관악구
15,플랜비 강남텐카페,서울시 강남구,NaN,서울시 강남구,NaN,NaN,NaN,NaN,강남구
17,쩜오 썸데이 한대표,서울 강남구 역삼동 831,서울특별시 강남구 역삼동 831,NaN,NaN,NaN,NaN,NaN,강남구


In [15]:
# sigu 생성

def build_sigu(gu):
    if pd.isna(gu):
        return np.nan
    gu = str(gu).strip()
    if not gu:
        return np.nan
    return f"서울특별시 {gu}"

df["sigu"] = df["gu"].apply(build_sigu)

# 점검
print("sigu 결측:", df["sigu"].isna().sum())
display(df[["gu","sigu"]].drop_duplicates().sort_values("gu").head(20))

sigu 결측: 1


,gu,sigu
0,강남구,서울특별시 강남구
1,관악구,서울특별시 관악구
3,서초구,서울특별시 서초구
6,송파구,서울특별시 송파구
12,중구,서울특별시 중구
5,NaN,NaN


In [16]:
clean_cols = [
    "name",
    "category",
    "address_std",
    "gu",
    "sigu",
    "lat",
    "lng",
    "is_valid_location",
    "COL_ADM_SE",
    "station",
    "station_exit",
    "station_detail",
]

df_clean = df[clean_cols].copy()

OUT = "../data/clean_roomhubs_seoul.csv"
df_clean.to_csv(OUT, index=False, encoding="utf-8-sig")

print("Saved:", OUT)
display(df_clean.head(20))

Saved: ../data/clean_roomhubs_seoul.csv


,name,category,address_std,gu,sigu,lat,lng,is_valid_location,COL_ADM_SE,station,station_exit,station_detail
0,달리는토끼(달토),룸싸롱/하퍼/셔츠룸/레깅스룸,서울특별시 강남구 역삼동 604-11,강남구,서울특별시 강남구,37.505834,127.031316,True,11230,NaN,NaN,NaN
1,Gallery,가라오케,서울특별시 관악구 신림동 1640-30,관악구,서울특별시 관악구,37.483453,126.928567,True,11210,NaN,NaN,NaN
2,달리는토끼,룸싸롱/하퍼/셔츠룸/레깅스룸,서울특별시 강남구 봉은사로 150,강남구,서울특별시 강남구,37.506122,127.031181,True,11230,NaN,NaN,NaN
3,BAR11,BAR/위스키바,서울특별시 서초구 서초대로78길 46,서초구,서울특별시 서초구,37.493620,127.028535,True,11220,NaN,NaN,NaN
4,쩜오 구구단 악어대표,쩜오,서울특별시 강남구 테헤란로 111 대건빌딩 지하1층,강남구,서울특별시 강남구,37.498720,127.029361,True,11230,NaN,NaN,NaN
5,다국적 노래클럽,노래주점/유흥주점,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,서울 등촌역삼거리
6,헤븐납득이,룸싸롱/하퍼/셔츠룸/레깅스룸,서울특별시 송파구 가락동 74,송파구,서울특별시 송파구,37.496511,127.121105,True,11240,NaN,NaN,NaN
7,Boss,룸싸롱/하퍼/셔츠룸/레깅스룸,서울특별시 송파구 오금로11길 29-14,송파구,서울특별시 송파구,37.515867,127.110384,True,11240,NaN,NaN,NaN
8,일프로/텐.쩜오 라임미녀실장,일프로/텐프로/텐카페,서울특별시 강남구 테헤란로 111,강남구,서울특별시 강남구,37.498893,127.029268,True,11230,NaN,NaN,NaN
9,핫플레이스 하퍼비트,룸싸롱/하퍼/셔츠룸/레깅스룸,서울특별시 강남구 삼성동 145,강남구,서울특별시 강남구,37.512803,127.053738,True,11230,NaN,NaN,NaN


In [17]:
# 품질 보고
print("총 행:", len(df))
print("실제 업소 주소(TRUE):", df["is_valid_location"].sum())
print("역 기반 주소(FALSE):", (~df["is_valid_location"]).sum())
print("최종 위경도 확보:", df["lat"].notna().sum())
print("위경도 결측:", df["lat"].isna().sum())
print("gu 결측:", df["gu"].isna().sum())

print("\n🔎 역 기반인데 address_std가 있는 케이스 샘플")
display(df[(df["is_valid_location"]==False) & (df["address_std"].notna())][
    ["name","address_raw","address_std","station_detail","station"]
].head(20))


총 행: 23
실제 업소 주소(TRUE): 19
역 기반 주소(FALSE): 4
최종 위경도 확보: 19
위경도 결측: 4
gu 결측: 1

🔎 역 기반인데 address_std가 있는 케이스 샘플


,name,address_raw,address_std,station_detail,station
11,신림다국적,서울 관악구 신림역사거리,서울특별시 관악구 신림역사거리,NaN,NaN
17,쩜오 썸데이 한대표,서울 강남구 역삼동 831,서울특별시 강남구 역삼동 831,NaN,NaN
